### In this notebook you will decode a baseband timestream from a dummy antenna to obtain the image encoded on a carrier wave.
We'll follow the FIR filtering notebook

In [ ]:
from matplotlib import pyplot as plt
import numpy as np
from scipy.signal import firwin

In [ ]:
### This is what's given to us about the baseband data we collected with our Car antenna.

In [ ]:

cfg = {}
cfg['dac_rate'] = 5e6 #Hz, sampling rate of output to DAC
cfg['freq_carrier'] = 1e6 #Hz, carrier frequency
cfg['code_baudrate'] = 200e3 #Hz, sampling rate for code. a new bit every 1/baudrate sec

#and a demod function, that you can ignore.

def demod(y, thresh=1e-3):
    # y is the downconverted, filtered, and downsampled array, sampled at 200 KSPS
    stretch = 6
    idx = np.abs(y)>1e-3
    recovered_bits = y[idx][3::stretch]>0 #get rid of bit repeats, stay away from edges
    reconstructed_array = np.packbits(recovered_bits)
    raw_byte_data = reconstructed_array.tobytes()
    with open('recovered_image.jpg', 'wb') as f:
        f.write(raw_byte_data)

In [ ]:
#read the baseband data
data = np.load('../data/image_baseband.npy')

In [ ]:
#visualize some parts of the data. Do you see the carrier wave?
# plt.plot()

In [ ]:
# design a filter to extract our signal
filter_size = 256
cutoff = 200e3/cfg['dac_rate'] # 200 kHz wide on each side, full width 400 kHz. you can play with it
print("filter +ve freq cutoff in units of sampling freq", cutoff)
h = firwin(filter_size, cutoff=cutoff, window='hamming',fs=1)
fig=plt.gcf()
fig.set_size_inches(12,2)
plt.subplot(121)
plt.title("Filter coefficients")
plt.plot(h)

plt.subplot(122)
plt.title("Filter response power")
filter_response = np.fft.fft( np.hstack([ h, np.zeros( 10*len(h) ) ]) ) 
#stack some zeros to increase freq resolution
freqs = np.fft.fftfreq(len(filter_response))
plt.semilogy( np.fft.fftshift(freqs),  np.fft.fftshift(np.abs(filter_response)**2))
plt.xlim(-cutoff-0.02, cutoff+0.02)
plt.axvline(cutoff, ls='dashed', c='black')
plt.axvline(-cutoff, ls='dashed', c='black',label='cutoff')
plt.axhline(0.25,ls='dotted', c='red',label='-6 dB point')
plt.xlabel("Freqs (fraction of Nyquist)")
plt.ylim(1e-7,1)
plt.legend()


### Downconvert

In [ ]:
x_dc = data*np.exp(-2j*np.pi*cfg['freq_carrier']*np.arange(len(data))/cfg['dac_rate'])
print(len(x_dc))

### Filter

In [ ]:
blocksize = 4096
nblocks = x_dc.shape[0]//blocksize
L = filter_size
x_dc = x_dc[:nblocks*blocksize].reshape(nblocks, blocksize) #snip the last remainder (none in our case)
print(x_dc.shape)
#create an input buffer with extra padding to put past data
inp = np.zeros((nblocks, blocksize + L), dtype='complex128')
inp[:, L:] = x_dc[:, :] #fill current
inp[1:, :L] = x_dc[:-1, -L:] #fill first padded chunk with past data data from filter-length-long bit of previous block

hf = np.fft.fft(np.hstack([h, np.zeros(blocksize)])) #create filter spectrum
filt_inp = np.fft.ifft( np.fft.fft(inp,axis=1) * hf[None,:], axis=1) #filter the input

#extract output by discarding output corresponding to padded blocks
out = filt_inp[:, L:]
out = np.ravel(out)

In [ ]:
out.shape

In [ ]:
dsamp = int(cfg['dac_rate']/cfg['code_baudrate'])
print("dsamp factor", dsamp)

In [ ]:
y = out.real[::dsamp]

### Visualize downsampled timestream without carrier. Do you see the code?

In [ ]:
plt.plot(y[:1000])

### Finally demodulate to extract the bits and save an image.

In [ ]:
demod(y)